In [ ]:
%pip install tensorflow

In [ ]:
# from google.colab import drive
import os
import pandas as pd

# 掛載 Google Drive
# drive.mount('/content/drive')

# 設定資料夾路徑
data_folder = 'data'

# 查看 data 資料夾中的檔案
print("data 資料夾中的檔案：")
files = os.listdir(data_folder)
for file in files:
    print(f"- {file}")

# 篩選出 CSV 檔案並讀取
csv_files = [f for f in files if f.endswith('.csv')]
print(f"\n找到 {len(csv_files)} 個 CSV 檔案")

# 讀取每個 CSV 檔案
dataframes = {}
for csv_file in csv_files:
    file_path = os.path.join(data_folder, csv_file)
    try:
        df = pd.read_csv(file_path)
        dataframes[csv_file] = df
        print(f"\n成功讀取 {csv_file}，形狀: {df.shape}")
        print(f"前 3 行預覽：")
        print(df.head(3))
    except Exception as e:
        print(f"讀取 {csv_file} 時發生錯誤: {e}")


In [ ]:
# 合併所有數據
all_data = pd.concat(list(dataframes.values()), ignore_index=True)

# 檢查合併後的數據形狀
print(f"合併後的數據形狀: {all_data.shape}")

# 檢查訂單狀態分佈
status_counts = all_data['StatusDef'].value_counts()
print("\n訂單狀態分佈:")
print(status_counts)
print(f"總筆數: {status_counts.sum()}")

# 創建目標變量：訂單是否失敗
# 失敗訂單包括：Cancel, Fail, Overdue
all_data['order_failed'] = all_data['StatusDef'].apply(
    lambda x: 1 if x in ['Cancel', 'Fail', 'Overdue'] else 0
)

# 檢查目標變量分佈
target_counts = all_data['order_failed'].value_counts()
print("\n訂單失敗/成功分佈:")
print(target_counts)
print(f"失敗訂單比例: {target_counts[1] / len(all_data):.4f}")


In [ ]:
# 轉換時間列為datetime格式
time_columns = ['OrderDateTime', 'BehaviorWindow_Start', 'BehaviorWindow_End']
for col in time_columns:
    all_data[col] = pd.to_datetime(all_data[col])

# 提取時間特徵
all_data['order_hour'] = all_data['OrderDateTime'].dt.hour
all_data['order_day'] = all_data['OrderDateTime'].dt.day_of_week  # 0=星期一, 6=星期日
all_data['order_month'] = all_data['OrderDateTime'].dt.month
all_data['behavior_window_hours'] = (all_data['BehaviorWindow_End'] - all_data['BehaviorWindow_Start']).dt.total_seconds() / 3600

# 檢查提取的時間特徵
print("時間特徵摘要統計:")
print(all_data[['order_hour', 'order_day', 'order_month', 'behavior_window_hours']].describe())

# 檢查 Age 欄位（根據文件說明，若生日是"1900-01-01"應視為空值）
# 先檢查 Age 的分佈
print("\nAge 欄位分佈:")
print(all_data['Age'].describe())
print(f"Age 欄位缺失值數量: {all_data['Age'].isna().sum()}")

# 處理 Age 欄位
# 如果有異常值（例如年齡過大或過小），可以將其視為缺失值
age_lower_bound = 10
age_upper_bound = 100
all_data.loc[(all_data['Age'] < age_lower_bound) | (all_data['Age'] > age_upper_bound), 'Age'] = np.nan


In [ ]:
# 處理類別特徵
categorical_features = [
    'ChannelType', 'ChannelDetail', 'PaymentType', 'ShippingType',
    'RegisterSourceTypeDef', 'Gender', 'CountryAliasCode'
]

# 檢查每個類別特徵的唯一值數量
print("類別特徵的唯一值數量:")
for col in categorical_features:
    print(f"{col}: {all_data[col].nunique()} 個唯一值")
    print(f"前5個值: {all_data[col].value_counts().head(5).to_dict()}")
    print(f"缺失值數量: {all_data[col].isna().sum()}")
    print("---")

# 填充缺失值
for col in categorical_features:
    all_data[col] = all_data[col].fillna('Unknown')

# 對類別特徵進行獨熱編碼
# 為了避免維度爆炸，我們可以只保留頻率較高的類別
for col in categorical_features:
    # 獲取該特徵的值計數
    value_counts = all_data[col].value_counts()
    # 保留頻率前10的類別，其他歸為'Other'
    top_categories = value_counts.nlargest(10).index.tolist()
    all_data[col] = all_data[col].apply(lambda x: x if x in top_categories else 'Other')

# 進行獨熱編碼
all_data_encoded = pd.get_dummies(
    all_data,
    columns=categorical_features,
    prefix=categorical_features,
    drop_first=False
)

# 檢查編碼後的數據形狀
print(f"編碼前數據形狀: {all_data.shape}")
print(f"編碼後數據形狀: {all_data_encoded.shape}")

# 查看新增的特徵列
new_columns = set(all_data_encoded.columns) - set(all_data.columns)
print(f"新增特徵數量: {len(new_columns)}")
print("前10個新增特徵:")
print(list(new_columns)[:10])


In [ ]:
# 處理數值特徵
numerical_features = [
    'Qty', 'TotalSalesAmount', 'TotalPrice', 'TotalDiscount',
    'Age', 'MemberCardLevel', 'behavior_window_hours',
    'order_hour', 'order_day', 'order_month'
]

# 檢查數值特徵的基本統計
print("數值特徵的基本統計:")
print(all_data_encoded[numerical_features].describe())

# 檢查缺失值
print("\n數值特徵的缺失值數量:")
for col in numerical_features:
    print(f"{col}: {all_data_encoded[col].isna().sum()} 個缺失值")

# 填充缺失值
for col in numerical_features:
    median_value = all_data_encoded[col].median()
    all_data_encoded[col] = all_data_encoded[col].fillna(median_value)

# 檢查異常值
print("\n數值特徵的異常值檢查:")
for col in numerical_features:
    q1 = all_data_encoded[col].quantile(0.25)
    q3 = all_data_encoded[col].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr
    lower_bound = q1 - 1.5 * iqr
    outliers = all_data_encoded[(all_data_encoded[col] > upper_bound) | (all_data_encoded[col] < lower_bound)]
    print(f"{col}: {len(outliers)} 個異常值 ({len(outliers) / len(all_data_encoded) * 100:.2f}%)")

# 處理異常值 - 使用上下限截斷
for col in numerical_features:
    q1 = all_data_encoded[col].quantile(0.25)
    q3 = all_data_encoded[col].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr
    lower_bound = q1 - 1.5 * iqr
    all_data_encoded[col] = all_data_encoded[col].clip(lower=lower_bound, upper=upper_bound)

# 標準化數值特徵
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
all_data_encoded[numerical_features] = scaler.fit_transform(all_data_encoded[numerical_features])

# 檢查標準化後的數值特徵
print("\n標準化後的數值特徵摘要統計:")
print(all_data_encoded[numerical_features].describe())


In [ ]:
# 提取行為特徵
behavior_columns = [col for col in all_data_encoded.columns if col.startswith('Behavior_')]
device_columns = [col for col in all_data_encoded.columns if col.startswith('Device_')]
tunnel_columns = [col for col in all_data_encoded.columns if col.startswith('Tunnel_')]

# 合併所有行為相關特徵
behavior_features = behavior_columns + device_columns + tunnel_columns

print(f"行為特徵數量: {len(behavior_features)}")
print("行為特徵列表:")
print(behavior_features)

# 檢查行為特徵的分佈
print("\n行為特徵摘要統計:")
behavior_stats = all_data_encoded[behavior_features].describe().transpose()[['mean', 'std', 'max']]
print(behavior_stats.head(10))  # 只顯示前10個行為特徵的統計

# 檢查行為特徵之間的相關性
import matplotlib.pyplot as plt
import seaborn as sns

# 計算行為特徵之間的相關性
behavior_corr = all_data_encoded[behavior_features].corr()

# 繪製熱力圖
plt.figure(figsize=(12, 10))
sns.heatmap(behavior_corr, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('行為特徵相關性熱力圖')
plt.tight_layout()
plt.show()

# 檢查行為特徵與目標變量的相關性
# 計算每個行為特徵與目標變量的相關係數
target_corr = {}
for feature in behavior_features:
    corr = all_data_encoded[feature].corr(all_data_encoded['order_failed'])
    target_corr[feature] = corr

# 按相關係數絕對值排序
sorted_corr = sorted(target_corr.items(), key=lambda x: abs(x[1]), reverse=True)

print("\n行為特徵與訂單失敗的相關性 (前15個):")
for feature, corr in sorted_corr[:15]:
    print(f"{feature}: {corr:.4f}")


In [ ]:
# 定義行為類型映射
behavior_types = {
    'Behavior_viewproduct': 1,      # 瀏覽商品頁
    'Behavior_search': 2,           # 搜尋
    'Behavior_add': 3,              # 加入購物車
    'Behavior_checkout': 4,         # 開始結帳
    'Behavior_purchase': 5,         # 購買
    'Behavior_selectcontent': 6,    # 點擊項目
    'Behavior_viewmainpage': 7,     # 首頁瀏覽
    'Behavior_viewcategory': 8,     # 分類頁瀏覽
    'Behavior_viewactivity': 9,     # 活動頁瀏覽
    'Behavior_viewcustompage': 10,  # 自訂頁瀏覽
    'Behavior_viewalbumdetail': 11, # 相簿模組內容頁瀏覽
    'Behavior_viewvideodetail': 12, # 影片模組內容頁瀏覽
    'Behavior_viewarticledetail': 13, # 文章模組內容頁瀏覽
    'Behavior_viewecoupondetail': 14, # 折價券詳細頁瀏覽
    'Behavior_viewpromotiondetail': 15, # 現折活動內容頁瀏覽
    'Behavior_viewcoupondetail': 16, # 優惠券內容頁瀏覽
    'Behavior_register': 17         # 會員註冊
}

# 定義設備類型映射
device_types = {
    'Device_Android APP': 1,
    'Device_Desktop': 2,
    'Device_MobileWeb': 3,
    'Device_iOS APP': 4
}

# 為每個訂單創建行為序列
def create_behavior_sequence(row, max_seq_length=20):
    """根據行為計數創建序列"""
    sequence = []

    # 遍歷所有行為類型
    for behavior, code in behavior_types.items():
        # 獲取該行為的計數
        count = int(row[behavior]) if behavior in row and not pd.isna(row[behavior]) else 0
        if count > 0:
            # 為每個計數添加一個行為點
            for _ in range(min(count, 5)):  # 限制每種行為最多5個點，避免序列過長
                # 確定設備類型 (選擇計數最高的設備)
                device_counts = [row[device] if device in row and not pd.isna(row[device]) else 0
                               for device in device_types.keys()]
                if sum(device_counts) > 0:
                    device_type = device_types[list(device_types.keys())[np.argmax(device_counts)]]
                else:
                    device_type = 0  # 如果沒有設備信息，使用0表示

                # 添加行為點 [行為類型, 設備類型, 相對時間]
                # 相對時間是根據行為窗口長度模擬的
                window_hours = row['behavior_window_hours'] if 'behavior_window_hours' in row and not pd.isna(row['behavior_window_hours']) else 72
                relative_time = np.random.uniform(0, window_hours)
                sequence.append([code, device_type, relative_time])

    # 按時間排序
    if sequence:
        sequence.sort(key=lambda x: x[2])

    # 處理序列長度
    if len(sequence) == 0:
        # 如果沒有行為記錄，創建一個全零序列
        sequence = np.zeros((max_seq_length, 3))
    elif len(sequence) < max_seq_length:
        # 如果序列太短，進行填充
        padding = np.zeros((max_seq_length - len(sequence), 3))
        sequence = np.vstack([sequence, padding])
    else:
        # 如果序列太長，只保留最近的行為
        sequence = sequence[-max_seq_length:]

    return np.array(sequence)

# 為了提高效率，我們只處理一部分數據
# 在實際應用中，您可能需要處理全部數據
sample_size = min(50000, len(all_data_encoded))  # 限制樣本大小
sample_data = all_data_encoded.sample(sample_size, random_state=42)

# 創建序列數據
print(f"為 {sample_size} 個訂單創建行為序列...")
sequences = []
for idx, row in sample_data.iterrows():
    if idx % 10000 == 0:
        print(f"處理第 {idx} 個訂單...")
    seq = create_behavior_sequence(row)
    sequences.append(seq)

# 轉換為numpy數組
X_sequences = np.array(sequences)

print(f"序列數據形狀: {X_sequences.shape}")
print("序列數據示例 (第一個訂單):")
print(X_sequences[0])

# 檢查序列數據的基本統計
sequence_lengths = np.sum(X_sequences[:, :, 0] > 0, axis=1)
print("\n序列長度統計:")
print(f"平均序列長度: {np.mean(sequence_lengths):.2f}")
print(f"最短序列長度: {np.min(sequence_lengths)}")
print(f"最長序列長度: {np.max(sequence_lengths)}")
print(f"序列長度中位數: {np.median(sequence_lengths)}")
print(f"空序列數量: {np.sum(sequence_lengths == 0)}")


In [ ]:
# 選擇靜態特徵
static_features = [
    # 訂單特徵
    'Qty', 'TotalSalesAmount', 'TotalPrice', 'TotalDiscount',
    'order_hour', 'order_day', 'order_month',

    # 用戶特徵
    'Age', 'MemberCardLevel',

    # 類別特徵 (編碼後)
    'ChannelType_N1Shopping', 'ChannelType_OfficialECom',
    'PaymentType_CreditCardOnce', 'PaymentType_LinePay', 'PaymentType_ATM',
    'ShippingType_Home', 'ShippingType_Oversea', 'ShippingType_SevenEleven',
    'Gender_Female', 'Gender_Male', 'Gender_Unknown',
    'CountryAliasCode_TW', 'CountryAliasCode_HK', 'CountryAliasCode_SG'
]

# 確保所有選擇的特徵都存在於數據框中
valid_static_features = [f for f in static_features if f in sample_data.columns]
if len(valid_static_features) < len(static_features):
    print(f"警告：有 {len(static_features) - len(valid_static_features)} 個靜態特徵不存在於數據框中")
    print(f"不存在的特徵：{set(static_features) - set(valid_static_features)}")
    # 如果有特徵不存在，可以考慮用相似的特徵替代或者忽略

# 提取靜態特徵
X_static = sample_data[valid_static_features].values

print(f"靜態特徵形狀: {X_static.shape}")
print("靜態特徵示例 (第一個訂單):")
print(X_static[0])
print("\n靜態特徵名稱:")
print(valid_static_features)

# 檢查靜態特徵的基本統計
print("\n靜態特徵基本統計:")
for i, feature in enumerate(valid_static_features):
    feature_values = X_static[:, i]
    print(f"{feature}: 平均值={np.mean(feature_values):.4f}, 標準差={np.std(feature_values):.4f}, 最小值={np.min(feature_values):.4f}, 最大值={np.max(feature_values):.4f}")


In [ ]:
# 提取目標變量
y = sample_data['order_failed'].values

# 檢查類別分佈
class_counts = np.bincount(y)
print("目標變量分佈:")
print(f"成功訂單 (0): {class_counts[0]} ({class_counts[0]/len(y)*100:.2f}%)")
print(f"失敗訂單 (1): {class_counts[1]} ({class_counts[1]/len(y)*100:.2f}%)")

# 劃分訓練集和測試集
from sklearn.model_selection import train_test_split

X_seq_train, X_seq_test, X_static_train, X_static_test, y_train, y_test = train_test_split(
    X_sequences, X_static, y, test_size=0.2, random_state=42, stratify=y
)

# 從訓練集中再劃分出驗證集
X_seq_train, X_seq_val, X_static_train, X_static_val, y_train, y_val = train_test_split(
    X_seq_train, X_static_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# 檢查數據劃分
print("\n數據劃分:")
print(f"訓練集: X_seq_train {X_seq_train.shape}, X_static_train {X_static_train.shape}, y_train {y_train.shape}")
print(f"驗證集: X_seq_val {X_seq_val.shape}, X_static_val {X_static_val.shape}, y_val {y_val.shape}")
print(f"測試集: X_seq_test {X_seq_test.shape}, X_static_test {X_static_test.shape}, y_test {y_test.shape}")

# 檢查類別分佈
print("\n類別分佈:")
print(f"訓練集: {np.bincount(y_train)}, 比例: {np.bincount(y_train) / len(y_train)}")
print(f"驗證集: {np.bincount(y_val)}, 比例: {np.bincount(y_val) / len(y_val)}")
print(f"測試集: {np.bincount(y_test)}, 比例: {np.bincount(y_test) / len(y_test)}")


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Concatenate, Bidirectional

# 獲取數據維度
seq_length = X_seq_train.shape[1]  # 序列長度，例如 20
seq_feature_dim = X_seq_train.shape[2]  # 序列特徵維度，例如 3
static_feature_dim = X_static_train.shape[1]  # 靜態特徵維度，例如 22

# 定義模型架構
# 1. 輸入層
sequence_input = Input(shape=(seq_length, seq_feature_dim), name='sequence_input')
static_input = Input(shape=(static_feature_dim,), name='static_input')

# 2. 序列處理分支
lstm_out = Bidirectional(LSTM(64, return_sequences=True))(sequence_input)
lstm_out = Dropout(0.3)(lstm_out)
lstm_out = Bidirectional(LSTM(32))(lstm_out)
lstm_out = Dropout(0.3)(lstm_out)

# 3. 靜態特徵處理分支
static_out = Dense(32, activation='relu')(static_input)
static_out = Dropout(0.2)(static_out)

# 4. 合併兩個分支
merged = Concatenate()([lstm_out, static_out])

# 5. 全連接層
dense = Dense(64, activation='relu')(merged)
dense = Dropout(0.3)(dense)
dense = Dense(32, activation='relu')(dense)
dense = Dropout(0.2)(dense)

# 6. 輸出層
output = Dense(1, activation='sigmoid')(dense)

# 7. 創建模型
model = Model(inputs=[sequence_input, static_input], outputs=output)

# 8. 編譯模型
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(), tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

# 9. 顯示模型摘要
model.summary()

# 10. 可視化模型架構
from tensorflow.keras.utils import plot_model
plot_model(model, to_file='order_risk_model.png', show_shapes=True, show_layer_names=True)


In [ ]:
# 設置早停策略
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# 學習率調度器
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=0.0001,
    verbose=1
)

# 模型檢查點
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_order_risk_model.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# 處理類別不平衡（因為失敗訂單比例只有12%）
# 計算類別權重
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# 計算類別權重
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

print("類別權重:", class_weight_dict)


In [ ]:
# 檢查數據類型
print("序列數據類型檢查:")
print(f"X_seq_train dtype: {X_seq_train.dtype}")
print(f"X_static_train dtype: {X_static_train.dtype}")
print(f"y_train dtype: {y_train.dtype}")

# 確保所有數據都是浮點數類型
X_seq_train = X_seq_train.astype(np.float32)
X_seq_val = X_seq_val.astype(np.float32)
X_seq_test = X_seq_test.astype(np.float32)

X_static_train = X_static_train.astype(np.float32)
X_static_val = X_static_val.astype(np.float32)
X_static_test = X_static_test.astype(np.float32)

# 確保標籤是整數類型
y_train = y_train.astype(np.int32)
y_val = y_val.astype(np.int32)
y_test = y_test.astype(np.int32)

# 再次檢查數據類型
print("\n修正後數據類型:")
print(f"X_seq_train dtype: {X_seq_train.dtype}")
print(f"X_static_train dtype: {X_static_train.dtype}")
print(f"y_train dtype: {y_train.dtype}")

# 檢查是否有無效值
print("\n檢查無效值:")
print(f"X_seq_train NaN數量: {np.isnan(X_seq_train).sum()}")
print(f"X_static_train NaN數量: {np.isnan(X_static_train).sum()}")
print(f"y_train NaN數量: {np.isnan(y_train).sum()}")

# 如果有NaN值，用0填充
if np.isnan(X_seq_train).sum() > 0:
    X_seq_train = np.nan_to_num(X_seq_train, nan=0.0)
    X_seq_val = np.nan_to_num(X_seq_val, nan=0.0)
    X_seq_test = np.nan_to_num(X_seq_test, nan=0.0)

if np.isnan(X_static_train).sum() > 0:
    X_static_train = np.nan_to_num(X_static_train, nan=0.0)
    X_static_val = np.nan_to_num(X_static_val, nan=0.0)
    X_static_test = np.nan_to_num(X_static_test, nan=0.0)

# 現在訓練模型
history = model.fit(
    [X_seq_train, X_static_train], y_train,
    validation_data=([X_seq_val, X_static_val], y_val),
    epochs=30,  # 設置足夠大的epoch數，早停策略會自動停止
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=[early_stopping, lr_scheduler, model_checkpoint],
    verbose=2
)

# 顯示訓練過程
print("訓練歷史:")
print(history.history.keys())


In [ ]:
# 下載台北思源黑體並命名taipei_sans_tc_beta.ttf，移至指定路徑
!wget -O TaipeiSansTCBeta-Regular.ttf https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_&export=download

import matplotlib
import matplotlib.pyplot as plt

# 添加字體
matplotlib.font_manager.fontManager.addfont('TaipeiSansTCBeta-Regular.ttf')
matplotlib.rc('font', family='Taipei Sans TC Beta')

# 首先檢查 history.history 中的所有鍵，以確定正確的指標名稱
print("訓練歷史中的所有指標:")
print(list(history.history.keys()))

# 繪製訓練和驗證損失
plt.figure(figsize=(15, 5))

# 損失曲線
plt.subplot(1, 3, 1)
plt.plot(history.history['loss'], label='訓練損失')
plt.plot(history.history['val_loss'], label='驗證損失')
plt.title('模型損失')
plt.xlabel('輪次')
plt.ylabel('損失')
plt.legend()

# 準確率曲線
plt.subplot(1, 3, 2)
plt.plot(history.history['accuracy'], label='訓練準確率')
plt.plot(history.history['val_accuracy'], label='驗證準確率')
plt.title('模型準確率')
plt.xlabel('輪次')
plt.ylabel('準確率')
plt.legend()

# AUC曲線 - 使用正確的鍵名
# 尋找包含 'auc' 的鍵
auc_key = [k for k in history.history.keys() if 'auc' in k and not k.startswith('val')][0]
val_auc_key = [k for k in history.history.keys() if 'auc' in k and k.startswith('val')][0]

plt.subplot(1, 3, 3)
plt.plot(history.history[auc_key], label='訓練AUC')
plt.plot(history.history[val_auc_key], label='驗證AUC')
plt.title('模型AUC')
plt.xlabel('輪次')
plt.ylabel('AUC')
plt.legend()

plt.tight_layout()
plt.savefig('training_metrics.png', dpi=300)
plt.show()

# 繪製精確率和召回率
plt.figure(figsize=(12, 5))

# 尋找包含 'precision' 和 'recall' 的鍵
precision_key = [k for k in history.history.keys() if 'precision' in k and not k.startswith('val')][0]
val_precision_key = [k for k in history.history.keys() if 'precision' in k and k.startswith('val')][0]
recall_key = [k for k in history.history.keys() if 'recall' in k and not k.startswith('val')][0]
val_recall_key = [k for k in history.history.keys() if 'recall' in k and k.startswith('val')][0]

# 精確率曲線
plt.subplot(1, 2, 1)
plt.plot(history.history[precision_key], label='訓練精確率')
plt.plot(history.history[val_precision_key], label='驗證精確率')
plt.title('模型精確率')
plt.xlabel('輪次')
plt.ylabel('精確率')
plt.legend()

# 召回率曲線
plt.subplot(1, 2, 2)
plt.plot(history.history[recall_key], label='訓練召回率')
plt.plot(history.history[val_recall_key], label='驗證召回率')
plt.title('模型召回率')
plt.xlabel('輪次')
plt.ylabel('召回率')
plt.legend()

plt.tight_layout()
plt.savefig('precision_recall_metrics.png', dpi=300)
plt.show()


In [ ]:
# 載入最佳模型
best_model = tf.keras.models.load_model('best_order_risk_model.h5')

# 在測試集上評估
test_results = best_model.evaluate([X_seq_test, X_static_test], y_test, verbose=1)
print(f"測試損失: {test_results[0]:.4f}")
print(f"測試準確率: {test_results[1]:.4f}")
print(f"測試AUC: {test_results[2]:.4f}")
print(f"測試精確率: {test_results[3]:.4f}")
print(f"測試召回率: {test_results[4]:.4f}")

# 計算F1分數
precision = test_results[3]
recall = test_results[4]
f1 = 2 * (precision * recall) / (precision + recall)
print(f"測試F1分數: {f1:.4f}")

# 預測
y_pred_prob = best_model.predict([X_seq_test, X_static_test])
y_pred = (y_pred_prob > 0.5).astype(int)

# 混淆矩陣
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# 確保使用台北思源黑體
matplotlib.font_manager.fontManager.addfont('TaipeiSansTCBeta-Regular.ttf')
matplotlib.rc('font', family='Taipei Sans TC Beta')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['成功', '失敗'],
            yticklabels=['成功', '失敗'])
plt.title('混淆矩陣')
plt.ylabel('真實標籤')
plt.xlabel('預測標籤')
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

# 分類報告
print("分類報告:")
print(classification_report(y_test, y_pred, target_names=['成功', '失敗']))


In [ ]:
# 創建一個簡化模型來分析特徵重要性
from tensorflow.keras.models import Model

# 提取靜態特徵的輸出
feature_importance_model = Model(inputs=static_input, outputs=static_out)

# 獲取靜態特徵的輸出值
feature_outputs = feature_importance_model.predict(X_static_test)

# 計算每個特徵的平均激活值
feature_importance = np.abs(feature_outputs).mean(axis=0)

# 將特徵重要性與特徵名稱對應
feature_importance_dict = dict(zip(valid_static_features, feature_importance))
sorted_importance = sorted(feature_importance_dict.items(), key=lambda x: x[1], reverse=True)

# 顯示特徵重要性
print("靜態特徵重要性 (前10個):")
for feature, importance in sorted_importance[:10]:
    print(f"{feature}: {importance:.4f}")

# 確保使用台北思源黑體
matplotlib.font_manager.fontManager.addfont('TaipeiSansTCBeta-Regular.ttf')
matplotlib.rc('font', family='Taipei Sans TC Beta')

# 繪製特徵重要性
plt.figure(figsize=(12, 8))
features = [x[0] for x in sorted_importance]
importances = [x[1] for x in sorted_importance]
plt.barh(features, importances)
plt.xlabel('重要性')
plt.title('靜態特徵重要性')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300)
plt.show()

# 分析行為序列對預測的影響
# 創建一個模型來獲取LSTM的輸出
lstm_output_model = Model(inputs=sequence_input, outputs=lstm_out)

# 獲取LSTM的輸出值
lstm_outputs = lstm_output_model.predict(X_seq_test)

# 檢查LSTM輸出的形狀
print(f"LSTM輸出形狀: {lstm_outputs.shape}")

# 根據實際形狀調整計算方法
if len(lstm_outputs.shape) == 2:
    # 如果LSTM輸出是2D的 (batch_size, features)
    # 這意味著我們使用的是最後一個LSTM層的return_sequences=False
    print("LSTM輸出是2D的，無法分析時間步重要性")

    # 我們可以分析特徵重要性
    lstm_feature_importance = np.abs(lstm_outputs).mean(axis=0)

    plt.figure(figsize=(10, 6))
    plt.bar(range(len(lstm_feature_importance)), lstm_feature_importance)
    plt.xlabel('LSTM特徵維度')
    plt.ylabel('重要性')
    plt.title('LSTM特徵重要性')
    plt.grid(True)
    plt.savefig('lstm_feature_importance.png', dpi=300)
    plt.show()

elif len(lstm_outputs.shape) == 3:
    # 如果LSTM輸出是3D的 (batch_size, time_steps, features)
    # 這意味著我們使用的是return_sequences=True

    # 計算每個時間步的平均激活值
    time_importance = np.abs(lstm_outputs).mean(axis=(0, 2))

    plt.figure(figsize=(10, 6))
    plt.plot(range(len(time_importance)), time_importance, marker='o')
    plt.xlabel('時間步')
    plt.ylabel('重要性')
    plt.title('序列時間步重要性')
    plt.grid(True)
    plt.savefig('time_importance.png', dpi=300)
    plt.show()

    # 也可以分析每個特徵的重要性
    feature_importance = np.abs(lstm_outputs).mean(axis=(0, 1))

    plt.figure(figsize=(10, 6))
    plt.bar(range(len(feature_importance)), feature_importance)
    plt.xlabel('LSTM特徵維度')
    plt.ylabel('重要性')
    plt.title('LSTM特徵重要性')
    plt.grid(True)
    plt.savefig('lstm_feature_importance.png', dpi=300)
    plt.show()

# 另一種分析方法：使用Permutation Feature Importance
# 這需要更多的計算資源，但提供更準確的特徵重要性
print("\n計算行為類型重要性...")

# 定義一個函數來擾動序列中的行為類型
def perturb_behavior_types(X_seq, behavior_idx=0):
    """
    擾動序列中的行為類型，用於評估行為類型的重要性

    參數:
    - X_seq: 行為序列數據
    - behavior_idx: 行為類型在序列特徵中的索引（通常是0）

    返回:
    - 擾動後的序列
    """
    X_seq_perturbed = X_seq.copy()

    # 對於非零行為，隨機替換行為類型
    mask = X_seq_perturbed[:, :, behavior_idx] > 0
    if np.sum(mask) > 0:
        # 獲取所有非零行為類型
        unique_behaviors = np.unique(X_seq[mask])
        if len(unique_behaviors) > 1:  # 確保有多個行為類型可供替換
            for i in range(X_seq_perturbed.shape[0]):
                for j in range(X_seq_perturbed.shape[1]):
                    if X_seq_perturbed[i, j, behavior_idx] > 0:
                        # 選擇一個不同的行為類型
                        current = X_seq_perturbed[i, j, behavior_idx]
                        others = [b for b in unique_behaviors if b != current]
                        X_seq_perturbed[i, j, behavior_idx] = np.random.choice(others)

    return X_seq_perturbed

# 由於完整的Permutation Feature Importance計算可能需要較長時間
# 這裡我們只使用一小部分測試數據進行演示
sample_size = min(1000, len(X_seq_test))
X_seq_sample = X_seq_test[:sample_size]
X_static_sample = X_static_test[:sample_size]
y_sample = y_test[:sample_size]

# 計算原始性能
y_pred_orig = best_model.predict([X_seq_sample, X_static_sample])
from sklearn.metrics import roc_auc_score
orig_auc = roc_auc_score(y_sample, y_pred_orig)
print(f"原始AUC: {orig_auc:.4f}")

# 擾動行為類型並計算性能下降
X_seq_perturbed = perturb_behavior_types(X_seq_sample)
y_pred_perturbed = best_model.predict([X_seq_perturbed, X_static_sample])
perturbed_auc = roc_auc_score(y_sample, y_pred_perturbed)
print(f"擾動後AUC: {perturbed_auc:.4f}")
print(f"行為類型重要性: {orig_auc - perturbed_auc:.4f}")

# 可以類似地分析設備類型和時間信息的重要性
# 但這裡為了節省時間，我們只演示了行為類型
